In [1]:
import pandas as pd
import numpy as np
import scipy.stats

import seaborn as sns

In [2]:
ratings = pd.read_csv('ratings.csv')

ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [3]:
# Number of users
print('The ratings dataset has', ratings['userId'].nunique(), 'unique users')

# Number of movies
print('The ratings dataset has', ratings['movieId'].nunique(), 'unique movies')

# Number of ratings
print('The ratings dataset has', ratings['rating'].nunique(), 'unique ratings')

# List of unique ratings
print('The unique ratings are', sorted(ratings['rating'].unique()))

The ratings dataset has 610 unique users
The ratings dataset has 9724 unique movies
The ratings dataset has 10 unique ratings
The unique ratings are [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]


In [4]:
movies = pd.read_csv('movies.csv')

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


--> Merging the two datasets to obtain a big dataset

In [5]:
df = pd.merge(ratings, movies, on='movieId', how='inner')

df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


--> Filtering out the movies which have more than 100 ratings

In [6]:
agg_ratings = df.groupby('title').agg(mean_rating = ('rating', 'mean'), number_of_ratings = ('rating', 'count')).reset_index()

agg_ratings_GT100 = agg_ratings[agg_ratings['number_of_ratings']>100]

agg_ratings_GT100

,title,mean_rating,number_of_ratings
64,2001: A Space Odyssey (1968),3.894495,109
191,Ace Ventura: Pet Detective (1994),3.040373,161
279,Aladdin (1992),3.792350,183
306,Alien (1979),3.969178,146
312,Aliens (1986),3.964286,126
...,...,...,...
8741,"Usual Suspects, The (1995)",4.237745,204
8830,WALL·E (2008),4.057692,104
8910,Waterworld (1995),2.913043,115
9094,Willy Wonka & the Chocolate Factory (1971),3.873950,119


--> Merging into the original dataset to make it have only movies which have ratings>100

In [7]:
df_GT100 = pd.merge(df, agg_ratings_GT100[['title']], on='title', how='inner')

df_GT100

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
2,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
3,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
4,1,110,4.0,964982176,Braveheart (1995),Action|Drama|War
...,...,...,...,...,...,...
19783,610,48516,5.0,1479542152,"Departed, The (2006)",Crime|Drama|Thriller
19784,610,58559,4.5,1493844688,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX
19785,610,60069,4.5,1493844866,WALL·E (2008),Adventure|Animation|Children|Romance|Sci-Fi
19786,610,68954,3.5,1493844881,Up (2009),Adventure|Animation|Children|Drama


In [8]:
# Number of users
print('The ratings dataset has', df_GT100['userId'].nunique(), 'unique users')

# Number of movies
print('The ratings dataset has', df_GT100['movieId'].nunique(), 'unique movies')

# Number of ratings
print('The ratings dataset has', df_GT100['rating'].nunique(), 'unique ratings')

# List of unique ratings
print('The unique ratings are', sorted(df_GT100['rating'].unique()))

The ratings dataset has 597 unique users
The ratings dataset has 134 unique movies
The ratings dataset has 10 unique ratings
The unique ratings are [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]


--> Creating a User-Movie Matrix. Here, the columns = userId, as we wish to calculate the similarity between the movies on the basis of the reviews they have received in the original dataset. This matrix shows what movie has received what ratings from what user.

In [9]:
matrix = df_GT100.pivot_table(index='title', columns='userId', values='rating')

matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
2001: A Space Odyssey (1968),NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,...,NaN,NaN,5.0,NaN,NaN,5.0,NaN,3.0,NaN,4.5
Ace Ventura: Pet Detective (1994),NaN,NaN,NaN,NaN,3.0,3.0,NaN,NaN,NaN,NaN,...,NaN,2.0,NaN,2.0,NaN,NaN,NaN,3.5,NaN,3.0
Aladdin (1992),NaN,NaN,NaN,4.0,4.0,5.0,3.0,NaN,NaN,4.0,...,NaN,NaN,NaN,3.0,3.5,NaN,NaN,3.0,NaN,NaN
Alien (1979),4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5.0,NaN,NaN,4.0,3.0,4.0,NaN,4.5
Aliens (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.0,NaN,NaN,3.5,NaN,4.5,NaN,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Usual Suspects, The (1995)",5.0,NaN,NaN,NaN,4.0,1.0,4.5,5.0,NaN,NaN,...,5.0,5.0,NaN,NaN,NaN,4.5,NaN,4.5,NaN,4.0
WALL·E (2008),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,4.5
Waterworld (1995),NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0,NaN,NaN,...,NaN,3.0,NaN,3.0,NaN,NaN,3.0,3.0,3.0,NaN


--> Normalizing the data by subtracting the average rating from each rating. Here, mean rating of each movie is calculated, and then it is subtracted from the rating received by an individual user.

In [10]:
matrix_norm = matrix.subtract(matrix.mean(axis=1), axis=0)

matrix_norm

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
2001: A Space Odyssey (1968),NaN,NaN,NaN,NaN,NaN,NaN,0.105505,NaN,NaN,NaN,...,NaN,NaN,1.105505,NaN,NaN,1.105505,NaN,-0.894495,NaN,0.605505
Ace Ventura: Pet Detective (1994),NaN,NaN,NaN,NaN,-0.040373,-0.040373,NaN,NaN,NaN,NaN,...,NaN,-1.040373,NaN,-1.040373,NaN,NaN,NaN,0.459627,NaN,-0.040373
Aladdin (1992),NaN,NaN,NaN,0.20765,0.207650,1.207650,-0.792350,NaN,NaN,0.20765,...,NaN,NaN,NaN,-0.792350,-0.29235,NaN,NaN,-0.792350,NaN,NaN
Alien (1979),0.030822,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.030822,NaN,NaN,0.030822,-0.969178,0.030822,NaN,0.530822
Aliens (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.035714,NaN,NaN,-0.464286,NaN,0.535714,NaN,1.035714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Usual Suspects, The (1995)",0.762255,NaN,NaN,NaN,-0.237745,-3.237745,0.262255,0.762255,NaN,NaN,...,0.762255,0.762255,NaN,NaN,NaN,0.262255,NaN,0.262255,NaN,-0.237745
WALL·E (2008),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.942308,NaN,NaN,NaN,NaN,-0.057692,NaN,NaN,NaN,0.442308
Waterworld (1995),NaN,NaN,NaN,NaN,NaN,0.086957,NaN,0.086957,NaN,NaN,...,NaN,0.086957,NaN,0.086957,NaN,NaN,0.086957,0.086957,0.086957,NaN


--> Calculating the similarity between movies on the basis of the ratings they have received from each user. Therefore, item_similarity = similarity of movies

In [11]:
item_similarity = matrix_norm.T.corr()
item_similarity

title,2001: A Space Odyssey (1968),Ace Ventura: Pet Detective (1994),Aladdin (1992),Alien (1979),Aliens (1986),"Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)",American Beauty (1999),American History X (1998),American Pie (1999),Apocalypse Now (1979),...,True Lies (1994),"Truman Show, The (1998)",Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Twister (1996),Up (2009),"Usual Suspects, The (1995)",WALL·E (2008),Waterworld (1995),Willy Wonka & the Chocolate Factory (1971),X-Men (2000)
title,,,,,,,,,,,,,,,,,,,,,
2001: A Space Odyssey (1968),1.000000,-0.036319,0.017446,0.318523,0.317386,0.324150,0.193592,0.152405,0.011490,0.478877,...,-0.108291,-0.012451,-0.041791,-0.458642,0.152271,0.245279,0.100172,-0.447306,0.087803,-0.123862
Ace Ventura: Pet Detective (1994),-0.036319,1.000000,0.302193,-0.208017,-0.107524,-0.030425,0.040435,0.065549,0.173855,0.245829,...,0.139896,0.188089,0.054408,0.176930,-0.007853,-0.061520,0.170717,0.176155,0.051239,0.045676
Aladdin (1992),0.017446,0.302193,1.000000,0.026514,0.151152,0.445204,0.127764,0.262014,0.367076,0.015038,...,0.333687,0.562311,-0.069176,0.137215,0.171330,0.153934,0.272375,0.065342,0.164459,0.285480
Alien (1979),0.318523,-0.208017,0.026514,1.000000,0.705925,0.387215,0.215751,0.035373,-0.006804,0.378709,...,0.199538,0.178620,0.108327,0.022007,-0.098813,0.350428,0.270697,0.119849,0.117749,0.030257
Aliens (1986),0.317386,-0.107524,0.151152,0.705925,1.000000,0.540458,0.111452,0.139326,0.076674,0.221920,...,0.369971,0.287243,0.084792,0.092412,0.195581,0.296933,0.294852,-0.014274,0.111864,0.225923
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Usual Suspects, The (1995)",0.245279,-0.061520,0.153934,0.350428,0.296933,0.334559,0.350545,0.276079,0.280924,0.182520,...,0.143424,0.064293,0.255347,-0.390032,0.468998,1.000000,0.378580,-0.194911,0.220837,0.173727
WALL·E (2008),0.100172,0.170717,0.272375,0.270697,0.294852,0.102326,0.216398,0.042043,0.060699,0.165662,...,0.303725,0.138136,0.513187,-0.162762,0.496807,0.378580,1.000000,-0.112344,0.095842,0.126949
Waterworld (1995),-0.447306,0.176155,0.065342,0.119849,-0.014274,-0.322291,-0.216045,-0.187477,0.180433,-0.094090,...,0.092716,0.141998,0.252467,0.191565,-0.200842,-0.194911,-0.112344,1.000000,0.136267,0.115242


--> Predict user's rating for one movie: Finding the similarity b/w the movies rated by the user and a movie the user has not rated yet. This is an example to show how the item-based CF system can help with predicting the ratings of unwatched movies

In [ ]:
# Picking a user
picked_user_id = 1

# Picking a movie
picked_movie = 'American Pie (1999)'

# Obtaining all the movies the user has rated
picked_userid_watched = pd.DataFrame(matrix_norm[picked_user_id].dropna(axis=0, how='all').sort_values(ascending=False).reset_index().rename(columns={1:'rating'}))

picked_userid_watched.shape
picked_userid_watched

,title,rating
0,Dumb & Dumber (Dumb and Dumber) (1994),1.939850
1,Indiana Jones and the Temple of Doom (1984),1.361111
2,X-Men (2000),1.300752
3,E.T. the Extra-Terrestrial (1982),1.233607
4,Ghostbusters (a.k.a. Ghost Busters) (1984),1.225000
5,Willy Wonka & the Chocolate Factory (1971),1.126050
6,"Terminator, The (1984)",1.103053
7,"Big Lebowski, The (1998)",1.075472
8,Gladiator (2000),1.061765
9,Seven (a.k.a. Se7en) (1995),1.024631


In [27]:
# Now, we need to find the similarity b/w each movie rated by the user and the picked movie
# For this, we have the picked movie. So we will fetch the similarities of all movies with the picked movie = picked_movie_similarity_score.

# Similarity score of the picked movie (American Pie) and all the movies
picked_movie_similarity_score = item_similarity[[picked_movie]].reset_index().rename(columns={'American Pie (1999)': 'similarity_score'})

print(picked_movie_similarity_score.shape)

n = 5

# Similarity score of the picked movie and all the movies rated by the user
# The dataset picked_movie_similarity_score contains the titles of all the movies which have been rated by the user + the similarity score with the picked movie
# So, we use the merge operation to filter out only the movies rated by the target user along with the similarity score.
picked_userid_watched_similarity = pd.merge(left=picked_userid_watched, right=picked_movie_similarity_score, on='title', how='inner').sort_values('similarity_score', ascending=False)

picked_userid_watched_similarity.shape

(134, 2)


(56, 3)

--> Calculating the predicted rating of the movie 'American Pie (1999)' by the user 1

In [14]:
predicted_rating = round(np.average(picked_userid_watched_similarity['rating'], weights=picked_userid_watched_similarity['similarity_score']), 6)

print(predicted_rating)

0.338739


--> Building a recommendation system

In [40]:
# Item-based recommendation function
def item_based_rec(picked_userid=1, number_of_similar_items=5, number_of_recommendations =3):
  import operator

  # Movies that the target user has not watched

  # Picking the rows from the matrix_norm where id = picked_userid. isna() = True implies that the value in that place has not been entered.
  picked_userid_unwatched = pd.DataFrame(matrix_norm[picked_userid].isna()).reset_index()
  # print(picked_userid_unwatched)
  # picking the movies where rating is NaN
  picked_userid_unwatched = picked_userid_unwatched[picked_userid_unwatched[1]==True]['title'].values.tolist()
  # print(picked_userid_unwatched)

  # Movies that the target user has watched
  picked_userid_watched = pd.DataFrame(matrix_norm[picked_userid].dropna(axis=0, how='all')\
                            .sort_values(ascending=False))\
                            .reset_index()\
                            .rename(columns={1:'rating'})
  
  # Dictionary to save the unwatched movie and predicted rating pair
  rating_prediction ={}  

  # Loop through unwatched movies          
  for picked_movie in picked_userid_unwatched: 

    # Calculate the similarity score of the picked movie with other movies
    # Fetching the similarity score of the picked movie with all the movies
    # This will give us all the data about the similarity of all the movies with that of the movie under consideration.
    # The picked movie will definitely be an unwatched movie, but the movie whose similarity is being shown with are watched + unwatched
    picked_movie_similarity_score = item_similarity[[picked_movie]].reset_index().rename(columns={picked_movie:'similarity_score'})

    # Rank the similarities between the picked user watched movie and the picked unwatched movie.
    
    # Now, we need to fetch the similarity b/w the watched and the unwatched movies
    # For this, we merge the previous dataset with the watched movies, as the former contains the unwatched, and the latter contains the watched
    picked_userid_watched_similarity = pd.merge(left=picked_userid_watched, 
                                                right=picked_movie_similarity_score, 
                                                on='title', 
                                                how='inner')\
                                        .sort_values('similarity_score', ascending=False)[:number_of_similar_items]
    
    # Calculate the predicted rating using weighted average of similarity scores and the ratings from user 1
    predicted_rating = round(np.average(picked_userid_watched_similarity['rating'], 
                                        weights=picked_userid_watched_similarity['similarity_score']), 6)
    
    # Save the predicted rating in the dictionary
    rating_prediction[picked_movie] = predicted_rating

  print("Unwatched Movies: ", picked_userid_unwatched)
  print("Watched Movies: ", picked_userid_watched.head())
  print("picked_movie_similarity_score", picked_movie_similarity_score.head())
  print("picked_userid_watched_similarity", picked_userid_watched_similarity.head())

  # Return the top recommended movies
  return sorted(rating_prediction.items(), key=operator.itemgetter(1), reverse=True)[:number_of_recommendations]

# Get recommendations
recommended_movie = item_based_rec(picked_userid=1, number_of_similar_items=5, number_of_recommendations =3)
recommended_movie


Unwatched Movies:  ['2001: A Space Odyssey (1968)', 'Ace Ventura: Pet Detective (1994)', 'Aladdin (1992)', 'Aliens (1986)', "Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)", 'American Pie (1999)', 'Apollo 13 (1995)', 'Austin Powers: The Spy Who Shagged Me (1999)', 'Babe (1995)', 'Batman Begins (2005)', 'Batman Forever (1995)', 'Beautiful Mind, A (2001)', 'Beauty and the Beast (1991)', 'Blade Runner (1982)', 'Bourne Identity, The (2002)', 'Breakfast Club, The (1985)', 'Catch Me If You Can (2002)', 'Cliffhanger (1993)', 'Clueless (1995)', 'Crimson Tide (1995)', 'Crouching Tiger, Hidden Dragon (Wo hu cang long) (2000)', 'Dark Knight, The (2008)', 'Departed, The (2006)', 'Die Hard (1988)', 'Die Hard: With a Vengeance (1995)', 'Donnie Darko (2001)', 'Eternal Sunshine of the Spotless Mind (2004)', "Ferris Bueller's Day Off (1986)", 'Fifth Element, The (1997)', 'Finding Nemo (2003)', 'Firm, The (1993)', 'Four Weddings and a Funeral (1994)', 'Ghost (1990)', 'Godfather, The (1972)', 'Godf

[('Austin Powers: The Spy Who Shagged Me (1999)', 1.096288),
 ('Crouching Tiger, Hidden Dragon (Wo hu cang long) (2000)', 0.92924),
 ('Lord of the Rings: The Return of the King, The (2003)', 0.926824)]